In [14]:
# RIN from the radius of a corrected I-Q circle.
# Based on Michaud-Belleau et al., Metrologia 53, 1154 (2016), equations (6a), (6b), and (7b).

from pathlib import Path
import numpy as np
import pandas as pd
try:
    import xy.pyplot as plt
except ImportError:
    import matplotlib.pyplot as plt
from scipy import signal

DATA_FILE = Path(r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\29_07_2026 HHI Data\HHI Coherent receiver\PPCL Whisper 700m delay\PPCL550_Whisper_14.66dBm_7.51+12.14dBm_no1.Wfm.csv")
SAMPLING_INTERVAL = None
AVERAGES = 20
MAX_PLOT_POINTS = 100_000
REMOVE_POLYNOMIAL_ORDER = 2


def metadata_sampling_interval(path):
    metadata_file = path.with_name(path.name[:-8] + ".csv") if path.name.endswith(".Wfm.csv") else None
    if metadata_file is None or not metadata_file.exists():
        raise FileNotFoundError("No matching metadata CSV found; set SAMPLING_INTERVAL manually.")
    for line in metadata_file.read_text(encoding="ISO-8859-1").splitlines():
        key, separator, value = line.partition(":")
        if separator and key.strip() == "SignalResolution":
            return float(value.strip().split(":")[0])
    raise KeyError(f"SignalResolution was not found in {metadata_file}")


def load_iq_csv(path, dt=None):
    frame = pd.read_csv(path, header=None, delimiter=";")
    if frame.shape[1] == 5:
        time = frame.iloc[:, 0].to_numpy(dtype=float)
        channels = frame.iloc[:, 1:5].to_numpy(dtype=float)
    elif frame.shape[1] == 4:
        channels = frame.to_numpy(dtype=float)
        dt = metadata_sampling_interval(path) if dt is None else dt
        time = np.arange(len(frame), dtype=float) * dt
    else:
        raise ValueError(f"Expected four channels or time plus four channels, got {frame.shape[1]} columns.")
    if len(time) < 2 or not np.all(np.isfinite(time)) or not np.all(np.isfinite(channels)):
        raise ValueError("The input must contain at least two finite samples.")
    return time, channels


def correct_iq_preserve_dc(i_signal, q_signal):
    i_signal = np.asarray(i_signal, dtype=float)
    q_signal = np.asarray(q_signal, dtype=float)
    dc_i, dc_q = np.mean(i_signal), np.mean(q_signal)
    i_ac, q_ac = i_signal - dc_i, q_signal - dc_q
    gain_i, gain_q = np.sqrt(np.mean(i_ac ** 2)), np.sqrt(np.mean(q_ac ** 2))
    if gain_i == 0 or gain_q == 0:
        raise ValueError("One I-Q channel has zero AC variance.")
    i_normalized, q_normalized = i_ac / gain_i, q_ac / gain_q
    sin_delta = np.clip(np.mean(i_normalized * q_normalized), -1.0, 1.0)
    cos_delta = np.sqrt(max(1.0 - sin_delta ** 2, np.finfo(float).eps))
    i_corrected = i_signal / gain_i
    q_corrected = (q_signal / gain_q - i_corrected * sin_delta) / cos_delta
    parameters = {"dc_i": dc_i, "dc_q": dc_q, "gain_i": gain_i, "gain_q": gain_q, "quadrature_error_deg": np.degrees(np.arcsin(sin_delta))}
    return i_corrected, q_corrected, parameters


def radius_to_rin(i_signal, q_signal, detrend_order=None):
    radius = np.hypot(i_signal, q_signal)
    reference_radius = np.mean(radius)
    if reference_radius <= 0 or not np.isfinite(reference_radius):
        raise ValueError("The mean I-Q radius is invalid.")
    alpha = radius / reference_radius - 1.0
    if detrend_order is not None:
        alpha = signal.detrend(alpha, type="constant")
        if detrend_order > 0:
            sample_index = np.arange(alpha.size, dtype=float)
            alpha -= np.polyval(np.polyfit(sample_index, alpha, detrend_order), sample_index)
    return radius, alpha, reference_radius


def rin_psd(alpha, sampling_rate, averages=20):
    segment_length = alpha.size // averages
    if segment_length < 8:
        raise ValueError("Too few samples for the requested number of averages.")
    return signal.welch(alpha, fs=sampling_rate, window="hann", nperseg=segment_length, detrend="constant", scaling="density", return_onesided=True)


time, raw_channels = load_iq_csv(DATA_FILE, dt=SAMPLING_INTERVAL)
sampling_rate = 1.0 / np.median(np.diff(time))
I_corrected, Q_corrected, correction = correct_iq_preserve_dc(raw_channels[:, 0], raw_channels[:, 1])
radius, alpha, average_radius = radius_to_rin(I_corrected, Q_corrected, REMOVE_POLYNOMIAL_ORDER)
frequency, rin_psd_values = rin_psd(alpha, sampling_rate, AVERAGES)
print(f"Loaded {len(time):,} samples from {DATA_FILE.name}")
print(f"Sampling rate: {sampling_rate:.6g} Hz")
print(f"Average corrected radius: {average_radius:.6g}")
print(f"RIN RMS: {np.std(alpha):.6g} ({20 * np.log10(np.std(alpha)):.2f} dBc RMS)")
print(f"Quadrature correction: {correction['quadrature_error_deg']:.3f} degrees")

plot_step = max(1, len(time) // MAX_PLOT_POINTS)
fig, axes = plt.subplots(2, 2, figsize=(15, 9), constrained_layout=True)
axes[0, 0].scatter(I_corrected[::plot_step], Q_corrected[::plot_step], s=2, alpha=0.2, linewidths=0)
axes[0, 0].set(xlabel="Corrected I", ylabel="Corrected Q", title="Corrected I-Q ring (points only)")
axes[0, 0].set_aspect("equal", adjustable="datalim")
axes[0, 0].grid(alpha=0.3)
axes[0, 1].plot(time[::plot_step], radius[::plot_step], linewidth=0.6, label="radius")
axes[0, 1].axhline(average_radius, color="black", linestyle="--", label="mean radius")
axes[0, 1].set(xlabel="Time (s)", ylabel="Radius", title=r"$r(t)=\sqrt{I_c^2+Q_c^2}$")
axes[0, 1].legend(); axes[0, 1].grid(alpha=0.3)
axes[1, 0].plot(time[::plot_step], alpha[::plot_step], linewidth=0.6)
axes[1, 0].set(xlabel="Time (s)", ylabel=r"$\alpha(t)=r(t)/\langle r\rangle-1$", title="Fractional intensity fluctuation")
axes[1, 0].grid(alpha=0.3)
valid_psd = frequency > 0
axes[1, 1].loglog(frequency[valid_psd], rin_psd_values[valid_psd], linewidth=1.2)
axes[1, 1].set(xlabel="Fourier frequency (Hz)", ylabel="RIN PSD (1/Hz)", title="Relative intensity noise")
axes[1, 1].grid(alpha=0.3, which="both")
plt.show()
output_file = DATA_FILE.with_name(DATA_FILE.stem.replace(".Wfm", "") + "_RIN_from_IQ.csv")
pd.DataFrame({"frequency_Hz": frequency, "RIN_PSD_1_per_Hz": rin_psd_values}).to_csv(output_file, index=False)
print(f"RIN PSD written to: {output_file}")

Loaded 20,000,000 samples from PPCL550_Whisper_14.66dBm_7.51+12.14dBm_no1.Wfm.csv
Sampling rate: 2e+07 Hz
Average corrected radius: 1.41566
RIN RMS: 0.0309893 (-30.18 dBc RMS)
Quadrature correction: -1.774 degrees


RIN PSD written to: C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\29_07_2026 HHI Data\HHI Coherent receiver\PPCL Whisper 700m delay\PPCL550_Whisper_14.66dBm_7.51+12.14dBm_no1_RIN_from_IQ.csv


In [15]:
# Reusable single-laser/single-delay batch analysis.
# Run one of the following configuration cells at a time.
# RIN is plotted in dBc/Hz: 10*log10(PSD of fractional intensity noise).

import gc

try:
    import xy.pyplot as plt
except ImportError:
    import matplotlib.pyplot as plt

BATCH_OUTPUT_ROOT = Path(r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31-07-2026 HHI Coherent receiver\RIN comparisons")
BATCH_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


def find_laser_family(path):
    text = str(path).lower()
    for family in ("whisper", "dither", "nkt", "agilent"):
        if family in text:
            return family.capitalize()
    return "Unknown"


def load_highfinesse_db(path):
    frame = pd.read_csv(path, encoding="ISO-8859-1", comment="#", dtype=np.float64)
    frequency_hz = pd.to_numeric(frame.iloc[:, 0], errors="coerce").to_numpy()
    db_per_hz = pd.to_numeric(frame.iloc[:, 1], errors="coerce").to_numpy()
    valid = np.isfinite(frequency_hz) & np.isfinite(db_per_hz) & (frequency_hz > 0) & (db_per_hz >= -300) & (db_per_hz <= 20)
    frequency_hz, db_per_hz = frequency_hz[valid], db_per_hz[valid]
    unique_frequency, unique_indices = np.unique(frequency_hz, return_index=True)
    return unique_frequency, db_per_hz[unique_indices]


def load_waveform_for_batch(path):
    frame = pd.read_csv(path, header=None, delimiter=";", dtype=np.float32, memory_map=True)
    if frame.shape[1] == 4:
        channels = frame.to_numpy(copy=True)
        dt = metadata_sampling_interval(path) if SAMPLING_INTERVAL is None else SAMPLING_INTERVAL
        time_data = np.arange(len(channels), dtype=np.float64) * dt
    else:
        time_data, channels = load_iq_csv(path, dt=SAMPLING_INTERVAL)
    del frame
    return time_data, channels


def run_rin_batch(laser_name, delay_m, data_folder, commercial_root, commercial_pattern="*RIN*.txt"):
    """Process all waveform captures for one laser and delay, then release memory."""
    data_folder, commercial_root = Path(data_folder), Path(commercial_root)
    output_folder = BATCH_OUTPUT_ROOT / laser_name / f"delay_{delay_m}m"
    output_folder.mkdir(parents=True, exist_ok=True)
    waveform_files = sorted(data_folder.glob("*.Wfm.csv"))
    commercial_files = sorted(commercial_root.rglob(commercial_pattern))
    if not waveform_files:
        print(f"Skipped {laser_name} {delay_m} m: no .Wfm.csv files in {data_folder}")
        return
    if not commercial_files:
        print(f"Skipped {laser_name} {delay_m} m: no commercial files in {commercial_root}")
        return

    print(f"Processing {laser_name}, {delay_m} m: {len(waveform_files)} waveform files and {len(commercial_files)} commercial files")
    for waveform_file in waveform_files:
        time_data, channels = load_waveform_for_batch(waveform_file)
        local_sampling_rate = 1.0 / np.median(np.diff(time_data))
        stem = waveform_file.stem.replace(".Wfm", "")
        results = {}
        try:
            for label, pair in {"X": (0, 1), "Y": (2, 3)}.items():
                i_corrected, q_corrected, _ = correct_iq_preserve_dc(channels[:, pair[0]], channels[:, pair[1]])
                radius, alpha, _ = radius_to_rin(i_corrected, q_corrected, REMOVE_POLYNOMIAL_ORDER)
                frequencies, psd = rin_psd(alpha, local_sampling_rate, AVERAGES)
                db_per_hz = 10 * np.log10(np.maximum(psd, np.finfo(float).tiny))
                results[label] = {"i": i_corrected, "q": q_corrected, "radius": radius, "alpha": alpha, "f": frequencies, "db": db_per_hz}
                pd.DataFrame({"frequency_Hz": frequencies, "RIN_dBc_per_Hz": db_per_hz}).to_csv(output_folder / f"{stem}_{label}_RIN.csv", index=False)

            ring_fig, ring_axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
            point_step = max(1, len(channels) // 50_000)
            for axis, label in zip(ring_axes[:2], ("X", "Y")):
                axis.scatter(results[label]["i"][::point_step], results[label]["q"][::point_step], s=1, alpha=0.12)
                axis.set_aspect("equal", adjustable="box")
                axis.set(xlabel="Corrected I", ylabel="Corrected Q", title=f"{label} polarization ring")
                axis.grid(alpha=0.25)
            ring_axes[2].hist([results["X"]["radius"], results["Y"]["radius"]], bins=150, density=True, histtype="step", linewidth=1.2, label=["X", "Y"])
            ring_axes[2].set(xlabel="I-Q radius", ylabel="Probability density", title="Radius distributions")
            ring_axes[2].legend(); ring_axes[2].grid(alpha=0.25)
            ring_fig.savefig(output_folder / f"{stem}_IQ_rings.png", dpi=250)
            plt.close(ring_fig)

            comparison_fig, comparison_axis = plt.subplots(figsize=(12, 7), constrained_layout=True)
            plot_values = []
            for label, color in (("X", "tab:blue"), ("Y", "tab:orange")):
                valid = np.isfinite(results[label]["f"]) & np.isfinite(results[label]["db"]) & (results[label]["f"] > 0)
                comparison_axis.semilogx(results[label]["f"][valid], results[label]["db"][valid], color=color, linewidth=1.5, label=f"I-Q {label}")
                plot_values.append(results[label]["db"][valid])
            for commercial_file in commercial_files:
                commercial_frequency, commercial_db = load_highfinesse_db(commercial_file)
                comparison_axis.semilogx(commercial_frequency, commercial_db, color="0.25", alpha=0.35, linewidth=0.9, label=f"HighFinesse {commercial_file.stem[-3:]}")
                plot_values.append(commercial_db)
                pd.DataFrame({"frequency_Hz": commercial_frequency, "RIN_dBc_per_Hz": commercial_db}).to_csv(output_folder / f"{commercial_file.stem}_commercial.csv", index=False)
            finite_values = np.concatenate(plot_values)
            finite_values = finite_values[np.isfinite(finite_values)]
            comparison_axis.set_ylim(np.min(finite_values) - 5, np.max(finite_values) + 5)
            comparison_axis.set(xlabel="Fourier frequency (Hz)", ylabel="RIN (dBc/Hz)", title=f"{laser_name}, {delay_m} m: {stem}")
            comparison_axis.grid(alpha=0.3, which="both")
            comparison_axis.legend(fontsize=9, ncol=2)
            comparison_fig.savefig(output_folder / f"{stem}_RIN_comparison_dBc_per_Hz.png", dpi=300)
            plt.close(comparison_fig)
            print(f"  Finished {waveform_file.name}")
        finally:
            del time_data, channels, results
            gc.collect()
    print(f"Finished batch: {laser_name}, {delay_m} m -> {output_folder}")

In [ ]:
# PPCL Whisper, 700 m delay
run_rin_batch(
    laser_name="PPCL Whisper",
    delay_m=700,
    data_folder=r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\29_07_2026 HHI Data\HHI Coherent receiver\PPCL Whisper 700m delay",
    commercial_root=r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31-07-2026 HHI Coherent receiver\PPCL550 Whisper\HighFinesse RIN",
    commercial_pattern="RIN*.txt",
)

In [ ]:
# PPCL Dither, 700 m delay
run_rin_batch(
    laser_name="PPCL Dither",
    delay_m=700,
    data_folder=r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\29_07_2026 HHI Data\HHI Coherent receiver\PPCL Dither 700m delay",
    commercial_root=r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31-07-2026 HHI Coherent receiver\PPCL550 Dither\HighFinesse RIN",
    commercial_pattern="RIN*.txt",
)

In [ ]:
# PPCL Whisper, 30 m delay
run_rin_batch(
    laser_name="PPCL Whisper",
    delay_m=30,
    data_folder=r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31-07-2026 HHI Coherent receiver\PPCL Whisper 30m",
    commercial_root=r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31-07-2026 HHI Coherent receiver\PPCL550 Whisper\HighFinesse RIN",
    commercial_pattern="RIN*.txt",
)

In [ ]:
# PPCL Dither, 30 m delay
run_rin_batch(
    laser_name="PPCL Dither",
    delay_m=30,
    data_folder=r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31-07-2026 HHI Coherent receiver\PPCL Dither 30m",
    commercial_root=r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31-07-2026 HHI Coherent receiver\PPCL550 Dither\HighFinesse RIN",
    commercial_pattern="RIN*.txt",
)

In [ ]:
# PPCL Whisper, 3 km delay
run_rin_batch(
    laser_name="PPCL Whisper",
    delay_m=3000,
    data_folder=r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31-07-2026 HHI Coherent receiver\PPCL Whisper 3km",
    commercial_root=r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31-07-2026 HHI Coherent receiver\PPCL550 Whisper\HighFinesse RIN",
    commercial_pattern="RIN*.txt",
)

In [ ]:
# PPCL Dither, 3 km delay
run_rin_batch(
    laser_name="PPCL Dither",
    delay_m=3000,
    data_folder=r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31-07-2026 HHI Coherent receiver\PPCL Dither 3km",
    commercial_root=r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31-07-2026 HHI Coherent receiver\PPCL550 Dither\HighFinesse RIN",
    commercial_pattern="RIN*.txt",
)

In [11]:
# NKT, 700 m delay
run_rin_batch(
    laser_name="NKT",
    delay_m=700,
    data_folder=r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31-07-2026 HHI Coherent receiver\NKT 700m delay",
    commercial_root=r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31-07-2026 HHI Coherent receiver\NKT\HighFinesse RIN",
    commercial_pattern="RIN*.txt",
)

Skipped NKT 700 m: no .Wfm.csv files in C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31-07-2026 HHI Coherent receiver\NKT 700m delay


In [ ]:
# NKT, 30 m delay
run_rin_batch(
    laser_name="NKT",
    delay_m=30,
    data_folder=r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31-07-2026 HHI Coherent receiver\NKT 14.66dBm 30m",
    commercial_root=r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31-07-2026 HHI Coherent receiver\NKT\HighFinesse RIN",
    commercial_pattern="RIN*.txt",
)

In [ ]:
# NKT, 3 km delay
run_rin_batch(
    laser_name="NKT",
    delay_m=3000,
    data_folder=r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31-07-2026 HHI Coherent receiver\NKT 14.66dBm 3km",
    commercial_root=r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31-07-2026 HHI Coherent receiver\NKT\HighFinesse RIN",
    commercial_pattern="RIN*.txt",
)

In [ ]:
# NKT, 10 km delay
run_rin_batch(
    laser_name="NKT",
    delay_m=10000,
    data_folder=r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31-07-2026 HHI Coherent receiver\NKT 10km",
    commercial_root=r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31-07-2026 HHI Coherent receiver\NKT\HighFinesse RIN",
    commercial_pattern="RIN*.txt",
)

In [ ]:
# Agilent 81940A, 700 m delay
run_rin_batch(
    laser_name="Agilent 81940A",
    delay_m=700,
    data_folder=r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31-07-2026 HHI Coherent receiver\Agilent 81940A 700m",
    commercial_root=r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31-07-2026 HHI Coherent receiver\Agilent 81940A\HighFinesse RIN",
    commercial_pattern="RIN*.txt",
)

In [ ]:
# Agilent 81940A, 30 m delay
run_rin_batch(
    laser_name="Agilent 81940A",
    delay_m=30,
    data_folder=r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31-07-2026 HHI Coherent receiver\Agilent 81940A 30m",
    commercial_root=r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31-07-2026 HHI Coherent receiver\Agilent 81940A\HighFinesse RIN",
    commercial_pattern="RIN*.txt",
)

In [ ]:
# Agilent 81940A, 3 km delay
run_rin_batch(
    laser_name="Agilent 81940A",
    delay_m=3000,
    data_folder=r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31-07-2026 HHI Coherent receiver\Agilent 81940A 3km",
    commercial_root=r"C:\Users\au617810\OneDrive - Aarhus universitet\O-drive - Jeppe\Coherent receiver\31-07-2026 HHI Coherent receiver\Agilent 81940A\HighFinesse RIN",
    commercial_pattern="RIN*.txt",
)